02613 course

64615


# Q1. What is the maximum wall time of the job script below?

**Svar**

2 hours

# Q2 The script below was successfully submitted, and it is now pending in the queue ‘hpc’, where the compute nodes have at most 768GB of physical memory.

**svar**

There will be a long pending time, because we request so much memory, and the scheduler needs to find and reserve suitable resources for the job

# Q3 A batch job produced the following job summary:

**Svar**

12 GB

# Q4 Consider the following profiling output obtained from running the script “process.py”:

In [7]:
total = 10.083
compute = 8.005
preprocess = 1.005

option1_runtime = (total - compute) + compute / 8
option1_speedup = total / option1_runtime

option2_runtime = (total - compute - preprocess) + (compute + preprocess) / 4
option2_speedup = total / option2_runtime

print(option1_runtime, option1_speedup)
print(option2_runtime, option2_speedup)

3.0786249999999997 3.27516342523042
3.3255 3.0320252593594947


**svar**

Option 1: parallel “compute” with 8 cores.


# Q5 Consider the following Python code that simulates how long a 1-dimensional random walk take to exit an interval given a starting position:

In [8]:
import random
import multiprocessing as mp

def escape_time(args):
    x0, std, l, u = args
    x = x0
    i = 0
    while l <= x <= u:
        x += random.gauss(mu=0.0, sigma=std)
        i += 1
    return i

tasks = [(0.5, 0.001, 0, 1) for _ in range(500)]

# Dynamic scheduling: workers receive tasks as they become available
with mp.Pool(processes=4) as pool:
    steps = list(pool.imap_unordered(escape_time, tasks, chunksize=1))

print(len(steps))
print(max(steps))
print(sum(steps) / len(steps))

Process SpawnPoolWorker-1:
Process SpawnPoolWorker-2:
Traceback (most recent call last):
Traceback (most recent call last):
  File "/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/multiprocessing/process.py", line 313, in _bootstrap
    self.run()
    ~~~~~~~~^^
  File "/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/multiprocessing/process.py", line 108, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/multiprocessing/pool.py", line 114, in worker
    task = get()
  File "/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/multiprocessing/queues.py", line 387, in get
    return _ForkingPickler.loads(res)
           ~~~~~~~~~~~~~~~~~~~~~^^^^^
AttributeError: Can't get attribute 'escape_time' on <module '__main__' (<class '_frozen_importlib.BuiltinImporter'>)>
  File "/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/multiprocessing/process.py", line 313, in _boot

KeyboardInterrupt: 

**Svar**


Dynamic scheduling

# Q6 Recall that 16-bit floating point numbers, as returned by NumPy’s “finfo”, have a resolution of 0.001, a minimum value of -6.55040e04 and a maximum value of 6.55040e04. Given the number a = 1000 in that format, what value will the following Python code print:

In [3]:
import numpy as np

a = np.float16(1000)
print(np.square(a))

inf


/var/folders/02/442vhy3x37zbc397sx5tbqt00000gn/T/ipykernel_65089/2617105538.py:4: RuntimeWarning: overflow encountered in square
  print(np.square(a))


**svar**

inf

# Q7 Recall that 16-bit floating point numbers, as returned by NumPy’s “finfo”, have a resolution of 0.001, a minimum value of -6.55040e04 and a maximum value of 6.55040e04. Given two numbers in that format, a = 10000 and b = 5, what value will be the output of the following Python statement:

In [5]:
a = np.float16(10000)
b = np.float16(5)

print(round(a+b))

10008


**svar**

10010

# Q8 We have a NumPy array “X”. The data type of “X” is float64, the shape is (3, 2) and the strides are (8, 24). Below are the contents of the 1D data buffer of the array:

In [15]:
import numpy as np

buffer = np.array([1, 2, 3, 4, 5, 6], dtype=np.float64)

X = np.ndarray(
    shape=(3, 2),
    dtype=np.float64,
    buffer=buffer,
    strides=(8, 24)
)

print(X)

[[1. 4.]
 [2. 5.]
 [3. 6.]]


**svar**

$
X = \begin{bmatrix} 1 & 4 \\ 2 & 5 \\ 3 & 6 \end{bmatrix}
$

# Q9 You have to compute the sum of a 3D array “X”. To maximize performance, you write the function using Numba for JIT compilation. After testing all possible loop orders, you find that the loop order that results in the best performance is

In [6]:
import numpy as np

shape = (800, 800, 800)
X = np.empty(shape, dtype=np.float64, order="F")  # example layout

# Manually checking the relevant stride order:
strides = (6400, 8, 5120000)

print(strides[1])  # stride for j, innermost loop

8


**svar**

`(6400, 8, 5120000)`

# Q10 Consider the following function: f(x, y) = x. Can we use f in a parallel reduction framework?

In [21]:
def check_associativity(f, a, b, c, name):
    left  = f(f(a, b), c)
    right = f(a, f(b, c))
    ok = abs(left - right) < 1e-9 if isinstance(left, float) else (left == right)
    print(f"{name}:")
    print(f"  f(f({a},{b}),{c}) = {left}")
    print(f"  f({a},f({b},{c})) = {right}")
    print(f"  Associative? {'✅ YES' if ok else '❌ NO — CANNOT use in parallel reduction'}")
    if not ok:
        print(f"  Fix: compute sum normally, apply |abs| at the end")
    print()

# abs(x+y) — NOT associative (exam 2024 Q6)
check_associativity(lambda x,y: abs(x+y), 1, 2, -3, "abs(x+y)")

# Regular sum — associative
check_associativity(lambda x,y: x+y, 1, 2, -3, "x+y (regular sum)")

# Max — associative
check_associativity(lambda x,y: max(x,y), 1, 5, 3, "max(x,y)")

# Set intersection — associative
def set_intersect(a, b): return a & b
A, B, C = {1,2,3}, {2,3,4}, {3,4,5}
left  = set_intersect(set_intersect(A,B), C)
right = set_intersect(A, set_intersect(B,C))
print(f"Set intersection:")
print(f"  (A∩B)∩C = {left}")
print(f"  A∩(B∩C) = {right}")
print(f"  Associative? {'✅ YES' if left == right else '❌ NO'}")

print()
print("SUMMARY:")
cases = [
    ("x + y (sum)",          "✅ Associative → OK for reduction"),
    ("x * y (product)",      "✅ Associative → OK for reduction"),
    ("max(x,y)",             "✅ Associative → OK for reduction"),
    ("min(x,y)",             "✅ Associative → OK for reduction"),
    ("Set intersection ∩",   "✅ Associative → OK for reduction"),
    ("abs(x+y)",             "❌ NOT associative → CANNOT use in reduction"),
    ("x - y (subtraction)",  "❌ NOT associative → CANNOT use in reduction"),
]
for name, verdict in cases:
    print(f"  {name:<30} {verdict}")

abs(x+y):
  f(f(1,2),-3) = 0
  f(1,f(2,-3)) = 2
  Associative? ❌ NO — CANNOT use in parallel reduction
  Fix: compute sum normally, apply |abs| at the end

x+y (regular sum):
  f(f(1,2),-3) = 0
  f(1,f(2,-3)) = 0
  Associative? ✅ YES

max(x,y):
  f(f(1,5),3) = 5
  f(1,f(5,3)) = 5
  Associative? ✅ YES

Set intersection:
  (A∩B)∩C = {3}
  A∩(B∩C) = {3}
  Associative? ✅ YES

SUMMARY:
  x + y (sum)                    ✅ Associative → OK for reduction
  x * y (product)                ✅ Associative → OK for reduction
  max(x,y)                       ✅ Associative → OK for reduction
  min(x,y)                       ✅ Associative → OK for reduction
  Set intersection ∩             ✅ Associative → OK for reduction
  abs(x+y)                       ❌ NOT associative → CANNOT use in reduction
  x - y (subtraction)            ❌ NOT associative → CANNOT use in reduction


In [20]:
def f(x, y):
    return x

x, y, z = 2, 5, 9

print("Commutative check:")
print(f(x, y), f(y, x))

print("Associative check:")
print(f(f(x, y), z), f(x, f(y, z)))

Commutative check:
2 5
Associative check:
2 2


**svar**

No, r is not commutative, but it is associative.

# Q11 Recall that the binary tree reduction has a theoretical runtime of log2(N) if we assume each operation (e.g., each addition) takes 1 time unit. In practice, to reduce overhead, we may do the reduction in chunks of 10 element at a time. Instead of each thread summing two elements at each level, it sums 10 elements. What is the theoretical run-time for such a chunked reduction?

In [27]:
import math

N = 1000000

binary_runtime = math.log2(N)
chunked_runtime = 10 * math.log10(N)

#math.log2(N) / 10 = 1.99

#math.log2(N/10) = 16.60

#math.log10(10 * N) = 7.0

print("Binary reduction:", binary_runtime)
print("Chunked reduction:", chunked_runtime)

Binary reduction: 19.931568569324174
Chunked reduction: 60.0


**Svar**

$ 10 \cdot log_{10}(N)$

# Q12 Given an n x n NumPy array, X, stored row-wise, we want to extract the upper k’th diagonal, for k >= 1. For example, for the array

In [63]:
X = np.array([
    [1,  2,  3,  4],
    [5,  6,  7,  8],
    [9, 10, 11, 12],
    [13,14, 15, 16]
])

n = X.shape[0]

for k in [1, 2]:
    print(X.reshape(-1)[k:-n*k:n+1])

[ 2  7 12]
[3 8]


**svar**

`X.reshape(-1)[k:-n*k:n+1]`

# Q13 Consider the following line of profiling output obtained by running the “kernprof” profiler on a function called “simulate” from a script “datasimulate.py”.

### FLOP/s from line profiler
1. Count FLOPs per loop iteration (each `+`, `-`, `*`, `/`, `sqrt` = 1 FLOP).
2. Multiply by number of iterations (`Hits`).
3. Divide by total time (convert to seconds).

```
a = x[i]*x[i] + 4      → 2 FLOPs
b = y[n-i-1] / x[i]   → 1 FLOP
z = z + a / b          → 2 FLOPs
Total: 5 FLOPs/iter × 10000 iters = 50000 FLOPs
Total time = 15000 µs = 0.015 s
→ 50000 / 0.015 = 3.33 × 10^6 FLOP/s
```


In [22]:
# FLOP/s calculator
flops_per_iter = 6   # mul+add, div, div+add
n_iters = 1000
total_flops = flops_per_iter * n_iters

# Total time: loop line + 3 body lines (in µs)
total_time_us = 166.0 + 232.0 + 158.0
total_time_s = total_time_us * 1e-3

flop_s = total_flops / total_time_s
print(f"FLOPs per iteration: {flops_per_iter}")
print(f"Total FLOPs: {total_flops:,}")
print(f"Total time: {total_time_s:.4f} s")
print(f"FLOP/s: {flop_s:.3e}")

FLOPs per iteration: 6
Total FLOPs: 6,000
Total time: 0.5560 s
FLOP/s: 1.079e+04


In [14]:
iterations = 1000
flops_per_iteration = 6

total_flops = iterations * flops_per_iteration

time_ms = 166 + 232 + 158
time_s = time_ms * 1e-3

flops_per_second = total_flops / time_s

print(flops_per_second)

10791.36690647482


**svar**

Roughly $10.8 \cdot 10^3$ FLOP/s

# Q14 Consider the following three functions: If we call both “localmax_rows” and localmax_cols” with a 2000 x 2000 NumPy array of 64-bit floating-point numbers stored row-wise, which function do we expect to have the shortest runtime? Assume w is set to 5.

**Svar**

`localmax_row`

# Q15 Consider the below CUDA kernel function that implements a local maximum filter


**Svar**

$1 \times 1024$

# Q16 Consider the job script below: If we want another job to wait until the job defined above has ended abnormally, what must we add to the job script of the other job?

**Svar**

add `#BSUB -w exit(process)`

# Q17 Consider the job script below defining a job array: Consider also the job script below, which defines a post processing job for the above job script: Given the following output from bjobs -A: When will the postprocessing job start?

**Svar**

Never

# Q18 Consider the following Python script: We run the script and it prints out ‘0.0’. We now run the script again. What will it print?

In [ ]:
import numpy as np

# Create the file with zeros once
x = np.memmap("mymemmap.raw", mode="w+", shape=(10, 10), dtype="float64")
x[:] = 0
x.flush()

# First run
x = np.memmap("mymemmap.raw", mode="r+", shape=(10, 10), dtype="float64")
print(x[0, 1])   # 0.0
x += 2
x.flush()

# Second run
x = np.memmap("mymemmap.raw", mode="r+", shape=(10, 10), dtype="float64")
print(x[0, 1])   # 2.0
x += 2
x.flush()

# Third run
x = np.memmap("mymemmap.raw", mode="r+", shape=(10, 10), dtype="float64")
print(x[0, 1])   # 2.0
x += 2
x.flush()

0.0
2.0
4.0


**Svar**

den stiger med 2 hver gang, så $2.0$

# Q19 We store these 3 images in a Zarr arrays.

**Svar**

The $1^{st}$ and the $3^{th}$

# Q20 Consider the following Python function which computes the sum over the columns of a provided NumPy array: Assume the arrays are stored row-wise. The performance of the code was measured for 4x4 and 8x8 matrices and was found to be 2 GFLOP/s. We now have a 65536 x 65536 matrix. If we ignore any overhead from the Python interpreter, we expect the performance:

In [5]:
X = np.zeros((65536, 65536), dtype=np.float64)

print(X.strides)

(524288, 8)


**Svar**

To be worse, i.e., less GFLOP/s

# Q21 Consider the following summary of a pandas DataFrame: We wish to reduce the size of the DataFrame. How might we reduce the size of the "date" column to the least amount of memory?

**Svar**


Convert to categorical data type

# Q22 The lazy software tester is asked to check the scalability ........   When asked by his manager about his findings, like scalability and time per dataset entry, which of the following statements the correct answer:



**Svar**

The data set I used is too small to make any conclusions about the scalability.  I need to look at a larger data set, as it seems that I have used more worker threads than entries in the set.  However, I can tell you that the dataset I used has between 9 and 12 entries, and that the compute time per entry is about 0.58 seconds.

# Q23 Consider the following Python-code: Which of the following statements is true?

**Svar**

The order of the loops should be swapped to access memory in a manner that utilizes caches efficiently

# Q24 Consider the following Python function, that calculates the sum over the diagonal elements of a matrix product of two quadratic matrices: You execute this code on a multi-core CPU.  Which of the following statements is true?

**Svar**

Multiple cores can be utilized with minimal effort, if the numpy module was built against a multi-threaded BLAS library and if we set the right environment variable to control the number of threads.